# 04 — Probabilistic Model Selection

This notebook compares nine pre-declared probabilistic
post-processing specifications.

Candidate models are fitted through four chronological expanding
folds. The holdout and external-test outcomes are not used.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import yaml

def locate_repository(start: Path) -> Path:
    current = start.resolve()

    for candidate in (current, *current.parents):
        if (
            candidate
            / "config/"
            "probabilistic_model_spec.yaml"
        ).exists():
            return candidate

    raise FileNotFoundError("Repository root not found.")

ROOT = locate_repository(Path.cwd())

completed = subprocess.run(
    [
        sys.executable,
        str(
            ROOT
            / "tools/"
            "run_probabilistic_oof_benchmark.py"
        ),
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)

print(completed.stdout)

if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError(
        "The probabilistic OOF benchmark failed."
    )


 PROBABILISTIC OOF MODEL SELECTION COMPLETE

Development folds: 4
Candidate specifications: 9
Quantiles per prediction: 99
Common support rows: 132
Common support dates: 38

Candidate ranking:
          candidate_model             family  mean_date_crps  standard_error_date_crps  relative_crps_reduction_vs_raw  available_oof_rows  within_one_standard_error  strict_crps_winner  parsimonious_selected_model
pooled_empirical_residual empirical_residual        0.732178                  0.076856                        0.619127                 152                       True                True                         True
  rule_empirical_residual empirical_residual        0.737334                  0.078549                        0.616445                 132                      False               False                        False
         rule_gp_matern32   gaussian_process        0.758982                  0.081732                        0.605184                 132                      F

## Candidate families

The nine numerical candidates represent five conceptual families:

1. raw deterministic forecast;
2. mean residual correction;
3. empirical residual distribution;
4. Gaussian process residual model;
5. CatBoost quantile regression.

Pooled and rule-specific versions test whether decision-rule
heterogeneity improves the forecast.

In [2]:
summary = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "04_candidate_score_summary.csv"
)

manifest = json.loads(
    (
        ROOT
        / "data/manifests/"
        "04_model_selection_manifest.json"
    ).read_text(encoding="utf-8")
)

print(
    summary[
        [
            "candidate_model",
            "family",
            "mean_date_crps",
            "standard_error_date_crps",
            "relative_crps_reduction_vs_raw",
            "within_one_standard_error",
            "strict_crps_winner",
            "parsimonious_selected_model",
        ]
    ].to_string(index=False)
)

          candidate_model             family  mean_date_crps  standard_error_date_crps  relative_crps_reduction_vs_raw  within_one_standard_error  strict_crps_winner  parsimonious_selected_model
pooled_empirical_residual empirical_residual        0.732178                  0.076856                        0.619127                       True                True                         True
  rule_empirical_residual empirical_residual        0.737334                  0.078549                        0.616445                      False               False                        False
         rule_gp_matern32   gaussian_process        0.758982                  0.081732                        0.605184                      False               False                        False
              rule_gp_rbf   gaussian_process        0.765892                  0.083071                        0.601589                      False               False                        False
 pooled_catboost_quantile

## Gaussian process interpretation

The Gaussian process models place a prior distribution over the
residual function.

The RBF and Matérn 3/2 functions are covariance kernels. They specify
different assumptions about how residual dependence declines as two
calendar inputs move apart. They are not probability kernels over
temperature events.

Observation noise is represented by a white-noise covariance term.
The predictive Gaussian distribution is converted to 99 quantiles.

## CRPS and the parsimony rule

CRPS evaluates the complete predictive distribution and has the same
physical unit as temperature. It rewards both concentration and
calibration.

The integral representation of CRPS is approximated using the 99
predicted quantiles.

The candidate with the smallest mean date-level CRPS is the strict
score winner. The final selected model is the least complex candidate
whose paired date-level CRPS difference from that winner is no larger
than one standard error.

In [3]:
score_panel = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "04_oof_score_panel.csv"
)

assert score_panel[
    "chronology_block"
].eq(
    "development_validation"
).all()

assert manifest["holdout_accessed"] is False
assert manifest["external_test_accessed"] is False

print(
    "Strict CRPS winner:",
    manifest["strict_crps_winner"],
)

print(
    "Parsimonious selected model:",
    manifest["selected_model"],
)

print(
    "Common support rows:",
    manifest["common_support_rows"],
)

print(
    "Common support dates:",
    manifest["common_support_dates"],
)

Strict CRPS winner: pooled_empirical_residual
Parsimonious selected model: pooled_empirical_residual
Common support rows: 132
Common support dates: 38


## Evidential boundary

This stage selects only the probabilistic specification. It does not
report holdout performance, June transfer performance, calibration
performance or trading profitability.

Those questions are addressed only after this selection is locked.